<a href="https://colab.research.google.com/github/Kareena-3/FlyRank-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

## Week 2 — My lane: CTR / Engagement Opportunity Scoring

This notebook frames the lane I chose in Week 1 as a concrete ML/data problem. I am using the small anonymized starter dataset that ships with this repository. The goal is decision-support: rank pages that look worth reviewing first, not claim that an edit will definitely cause a CTR improvement.


## 1. My lane as an ML task (type)

**Task type: scoring / ranking.**

The decision I want to improve is: **which visible content pages should a reviewer inspect first because they appear to under-capture clicks relative to pages in a similar search position?**

The output would be a ranked list of pages with an opportunity score and simple reason codes. A reviewer could then inspect the highest-ranked pages and decide whether a title/meta/snippet or intent/content change is appropriate.

I am calling this scoring/ranking rather than simple classification because the practical problem is prioritization: review capacity is limited, so the system should order candidates rather than only say yes/no.

**Who acts on it:** a content/SEO reviewer. **Action:** inspect the highest-ranked pages and decide whether a CTR-focused improvement is justified.

**Cost of a wrong recommendation:** a false positive can waste reviewer time or lead to an unnecessary edit; a false negative can leave a genuinely useful opportunity unreviewed. This is why minimum-volume checks and human review matter.

**Why data/ML can help:** CTR depends on context such as search position, impressions, content type, and engagement. A single fixed CTR cutoff cannot account for all of those signals. ML would only earn its place if it improves the ranking over a transparent baseline.


In [1]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# Make the notebook runnable in Google Colab and from the repository locally.
if "google.colab" in sys.modules:
    repo_dir = Path("/content/FlyRank-AI")
    if not repo_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/Kareena-3/FlyRank-AI.git",
            str(repo_dir)
        ], check=True)
    os.chdir(repo_dir)

data_path = Path("data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path.resolve()}")

df = pd.read_csv(data_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("Dataset:", data_path.resolve())


Rows: 30,000
Columns: 44
Dataset: /content/FlyRank-AI/data/raw/content_refresh_anonymized.csv


## 2. Target or proxy

The starter CSV is a 90-day snapshot, so it does **not** contain a clean future CTR outcome for a supervised prediction target. I therefore use a **current-window proxy** for this framing exercise: the gap between a page's observed CTR and the typical CTR for its `position_tier`.

For each page:

`ctr_gap = observed CTR - expected CTR for the same position tier`

and

`ctr_opportunity_proxy = -ctr_gap`

A larger positive proxy means the page's observed CTR is further below the position-tier average. This is an observed association/proxy, **not proof that changing the page will increase CTR**.

For a later supervised model, I would prefer a future-window target such as CTR change or recovery measured after the feature/decision window. That would avoid turning the current-window rule into the answer itself.


In [2]:
# Build a clean lane slice: enough impressions to reduce very-low-volume noise,
# and a valid position value (0 means no position data in this dataset).
lane_df = df[
    (df["impressions_90d"] >= 100) &
    (df["avg_position"] > 0)
].copy()

lane_df["expected_ctr"] = lane_df.groupby("position_tier")["ctr"].transform("mean")
lane_df["ctr_gap"] = lane_df["ctr"] - lane_df["expected_ctr"]
lane_df["ctr_opportunity_proxy"] = -lane_df["ctr_gap"]

print(f"Lane slice: {len(lane_df):,} pages")
print("\nProxy summary:")
print(lane_df["ctr_opportunity_proxy"].describe().round(4))

# Sketch the future target we would prefer once a time-series window is available.
lane_df["future_ctr_change_30d"] = np.nan
lane_df["future_target_available"] = False

print("\nTarget/proxy sketch:")
display(lane_df[[
    "content_id", "position_tier", "ctr", "expected_ctr",
    "ctr_gap", "ctr_opportunity_proxy", "future_ctr_change_30d"
]].head(10))


Lane slice: 22,006 pages

Proxy summary:
count    22006.0000
mean        -0.0000
std          0.3906
min        -11.4052
25%         -0.0742
50%          0.0924
75%          0.1941
max          0.3548
Name: ctr_opportunity_proxy, dtype: float64

Target/proxy sketch:


,content_id,position_tier,ctr,expected_ctr,ctr_gap,ctr_opportunity_proxy,future_ctr_change_30d
0,content_304f48230142,striking,0.76,0.255782,0.504218,-0.504218,NaN
1,content_a1fb4e703a9e,page_3_5,0.05,0.142359,-0.092359,0.092359,NaN
2,content_9aa793d4d895,page_3_5,0.09,0.142359,-0.052359,0.052359,NaN
3,content_331d6c4de07b,page_1,0.49,0.354760,0.135240,-0.135240,NaN
4,content_d99b7a2d90ca,page_3_5,0.13,0.142359,-0.012359,0.012359,NaN
5,content_d4084a4bc775,page_1,0.03,0.354760,-0.324760,0.324760,NaN
7,content_a63219c6e95a,page_3_5,0.06,0.142359,-0.082359,0.082359,NaN
8,content_5e6c160719bc,page_3_5,0.09,0.142359,-0.052359,0.052359,NaN
9,content_c27558df2b0c,page_1,0.16,0.354760,-0.194760,0.194760,NaN
10,content_d8ee6cc6d642,top_3,1.55,0.334128,1.215872,-1.215872,NaN


## 3. Success metric

**Primary success metric: Precision@20.**

The practical use case is a small review queue. Precision@20 asks: among the 20 pages the system ranks highest, how many are genuinely useful review candidates under the final outcome definition?

For the starter snapshot, I can inspect and rank the current proxy, but I should not pretend that this is a clean future supervised label. When the time-series warehouse is used later, I will define a future observed CTR/engagement outcome and use Precision@20 on that outcome. I will compare any ML ranking with a transparent position-adjusted baseline.

A good result therefore means **better top-of-list decisions**, not simply a high overall accuracy number.


In [3]:
# A transparent baseline ranking for the current snapshot:
# larger position-adjusted CTR opportunity first.
baseline_top20 = (
    lane_df.sort_values("ctr_opportunity_proxy", ascending=False)
    [["content_id", "content_type", "position_tier", "impressions_90d",
      "ctr", "expected_ctr", "ctr_opportunity_proxy"]]
    .head(20)
)

print("Top 20 pages under the transparent position-adjusted baseline:")
display(baseline_top20)


Top 20 pages under the transparent position-adjusted baseline:


,content_id,content_type,position_tier,impressions_90d,ctr,expected_ctr,ctr_opportunity_proxy
8939,content_478f26883850,comparison article,page_1,118,0.0,0.35476,0.35476
29977,content_c87291853cab,comparison article,page_1,112,0.0,0.35476,0.35476
29983,content_6880eb215048,keyword article,page_1,2845,0.0,0.35476,0.35476
8873,content_268b56dc0221,keyword article,page_1,130,0.0,0.35476,0.35476
8915,content_e0ae71489787,feedly article,page_1,381,0.0,0.35476,0.35476
33,content_d87a116e2c79,keyword article,page_1,298,0.0,0.35476,0.35476
2282,content_3d2f81868b16,keyword article,page_1,497,0.0,0.35476,0.35476
24588,content_dc0d776ba8e5,keyword article,page_1,310,0.0,0.35476,0.35476
11665,content_50c7fe0d308f,comparison article,page_1,161,0.0,0.35476,0.35476
11673,content_6b5196195b13,keyword article,page_1,329,0.0,0.35476,0.35476


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page/item.**

The starter dataset is already at page/content-item grain: each row represents one pseudonymized content item with trailing 90-day search and engagement measurements. For this lane I keep pages with at least 100 impressions and a valid average position so the ranking is less dominated by very-low-volume noise.

The important columns for this framing are `content_id` (identifier only), `content_type`, `position_tier`, `avg_position`, `impressions_90d`, `clicks_90d`, `ctr`, `sessions_90d`, and `engagement_rate`.


In [4]:
unit_columns = [
    "content_id", "content_type", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "sessions_90d",
    "engagement_rate"
]

print("One row = one pseudonymized content page/item")
display(lane_df[unit_columns].head(10))

print("\nRows in lane slice:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())


One row = one pseudonymized content page/item


,content_id,content_type,position_tier,avg_position,impressions_90d,clicks_90d,ctr,sessions_90d,engagement_rate
0,content_304f48230142,keyword article,striking,10.6,3803,29,0.76,17,5.88
1,content_a1fb4e703a9e,keyword article,page_3_5,20.3,15320,7,0.05,9,0.00
2,content_9aa793d4d895,keyword article,page_3_5,36.5,12581,11,0.09,11,0.00
3,content_331d6c4de07b,keyword article,page_1,6.2,11751,58,0.49,78,1.28
4,content_d99b7a2d90ca,keyword article,page_3_5,44.0,19140,24,0.13,145,0.00
5,content_d4084a4bc775,keyword article,page_1,8.5,3970,1,0.03,5,0.00
7,content_a63219c6e95a,keyword article,page_3_5,21.2,1724,1,0.06,28,3.57
8,content_5e6c160719bc,keyword article,page_3_5,46.0,32574,29,0.09,68,5.88
9,content_c27558df2b0c,keyword article,page_1,4.9,1240,2,0.16,3,0.00
10,content_d8ee6cc6d642,keyword article,top_3,2.2,20919,324,1.55,326,6.75



Rows in lane slice: 22006
Unique content IDs: 22006


## 5. Why ML beats a fixed rule here

A fixed rule such as `ctr < 0.5` is easy to explain, but it treats the same CTR threshold as equally meaningful across different search positions and contexts. Week 1 already showed that content types can also have materially different observed CTRs within the same position tier.

A learned scoring model could combine position, impressions, CTR, content type, freshness/age, engagement, and other observable signals and learn interactions between them. That is the reason ML may earn its place here.

However, I will **not assume ML is better**. I will first keep the transparent position-adjusted baseline, then test whether a learned ranking improves the top-K metric on a properly separated validation set. If the added model complexity does not improve the decision, the simpler rule should win.

The final output is decision-support: a ranked review queue with reason codes. It is not a causal claim that a particular edit will produce a particular CTR increase.


In [5]:
# Quick framing checks: the notebook should make the decision grain and proxy explicit.
print("Decision: which pages should be reviewed first for possible CTR opportunity?")
print("Unit of analysis: one pseudonymized content page/item")
print("Task type: scoring / ranking")
print("Primary metric: Precision@20 on a future observed outcome")
print("Current starter-data proxy: position-tier-adjusted CTR gap")
print("Lane slice rows:", len(lane_df))


Decision: which pages should be reviewed first for possible CTR opportunity?
Unit of analysis: one pseudonymized content page/item
Task type: scoring / ranking
Primary metric: Precision@20 on a future observed outcome
Current starter-data proxy: position-tier-adjusted CTR gap
Lane slice rows: 22006


## Self-check

- [x] Lane is explicitly named: CTR / Engagement Opportunity Scoring
- [x] Task type is scoring / ranking
- [x] Target/proxy is defined and its limitation is stated
- [x] Success metric is named before modeling: Precision@20
- [x] Unit of analysis is explicit: one row = one content page/item
- [x] A real dataframe and proxy columns are shown
- [x] The decision, action, and cost of a wrong recommendation are stated
- [x] ML is justified conditionally, not assumed to be better than a rule
- [x] Claims are framed as observed / directional / decision-support
- [x] No client names, raw URLs, or private queries are used


